# Estudio de Siniestralidad Vial en la Ciudad Autónoma de Buenos Aires
## Índice de Contenidos

1. **Introducción y Análisis Descriptivo Multidimensional**
   * 1.1 Diagnóstico de la Integración de Datasets (Carga y Fusión)
   * 1.2 Homogeneización Sintáctica y Normalización Estructurada
   * 1.3 Análisis de Siniestralidad según la Cronología Semanal
   * 1.4 Caracterización Demográfica y Matriz de Vulnerabilidad
   * 1.5 Dinámica de la Siniestralidad en Horas Pico (Extracción Cronológica)

2. **Ingeniería de Variables Spatiales y Modelado de Densidad Global**
   * 2.1 Vectorización Geográfica y GeoDataFrame Base
   * 2.2 Microanálisis Cartográfico e Interactividad Dinámica
   * 2.3 Delimitación de la Base de Siniestralidad Prioritaria (Filtro Crítico)
   * 2.4 Fundamentos de Geodesia: Coordenadas Planas (Gauss-Kruger CABA) vs. Geográficas (WGS84)
   * 2.5 Análisis Espacial Comparativo por Tipología de Actor Vial (Facet Grid)
   * 2.6 Estratificación Espacial mediante Rupturas Naturales de Jenks
   * 2.7 Análisis Jurisdiccional: Distribución Poligonal por Comunas
   * 2.8 Configuración y Diagnóstico del Modelo de Densidad Global DBSCAN
   * 2.9 Análisis de Sensibilidad: Escenario A (Enfoque Quirúrgico de Alta Densidad)
   * 2.10 Análisis de Sensibilidad: Escenario B (Enfoque Macrorregional Barrial)
   * 2.11 Cartografía de Alta Resolución y Aislamiento de Puntos Negros Críticos
   * 2.12 Regionalización Exhaustiva y Límites del Particionamiento K-Means

3. **Comparación de Modelos, Indexación Avanzada e Impacto en la Gestión Pública**
   * 3.1 Análisis de Sensibilidad Dinámica: El Factor Espacio-Temporal (Noche vs. Horas Pico)
   * 3.2 Perfilado de Vulnerabilidad y Tipificación de Actores por Sector K-Means
   * 3.3 Matriz de Correspondencia Jurisdiccional y Asignación Operativa por Comunas
   * 3.4 Cartografía Estática y Morfología de la Regionalización K-Means
   * 3.5 Indexación Geoespacial Avanzada mediante Grillas Hexagonales H3 (Uber)
   * 3.6 Análisis de Patrones Espaciales Mediante Representación Coroplética H3 (Top 10)
   * 3.7 Integración de Datos Topológicos de Redes Urbanas (OpenStreetMap via OSMnX)
   * 3.8 Diagnóstico Causal: Correlación entre Siniestralidad y Complejidad Topológica de la Red

In [ ]:
import pandas as pd
import IPython.display as display
from google.colab import drive
drive.mount('/content/drive')

# 1. Introducción y Preprocesamiento de Datos

Este estudio tiene como objetivo analizar la distribución espacial y temporal de los siniestros viales graves en la Ciudad Autónoma de Buenos Aires (CABA) para optimizar la asignación de recursos de seguridad pública.

A continuación, se realiza la importación de las bases de datos oficiales de "HECHOS" (con la información geográfica y cronológica de los incidentes) y "VÍCTIMAS" (con el detalle cualitativo de los actores involucrados).

### 1. CARGA Y EXPLORACIÓN INICIAL DE DATOS

In [ ]:
# Definición de rutas y archivos del proyecto
ruta = '/content/drive/MyDrive/MCDAT/Proyecto/'
archivo_siniestros = 'siniestros_viales_hechos.xlsx'

# Carga de la base de hechos (componente espacio-temporal)
hechos = pd.read_excel(ruta + archivo_siniestros, sheet_name='HECHOS')

# Carga de la base de víctimas (componente cualitativo)
victimas = pd.read_csv(ruta + 'siniestros_viales_victimas.csv', sep=';', on_bad_lines='skip', low_memory=False)


print("\n--- Vista Previa HECHOS ---")
print(hechos.head(3))

print("\n--- Vista Previa VICTIMAS ---")
print(victimas.head(3))

### 2. INTEGRACIÓN Y LIMPIEZA DE DATASETS

In [ ]:
import numpy as np

# Fusión interna mediante el id del siniestro
df_final = pd.merge(hechos, victimas, on='id_siniestro', how='inner')

# Estandarización de valores nulos
df_final = df_final.replace(['SD', 'sd', 'SD ', 'sd '], np.nan)

# Resolución y renombrado
df_final = df_final.rename(columns={
    'fecha_siniestro_x': 'fecha',
    'anio_siniestro_x': 'anio',
    'modo_desplazamiento_victima_x': 'modo_transporte'
})

# Eliminación de columnas duplicadas
columnas_duplicadas = ['fecha_siniestro_y', 'anio_siniestro_y', 'modo_desplazamiento_victima_y','rango_horario']
df_final = df_final.drop(columns=columnas_duplicadas, errors='ignore')

# Control de consistencia del dataset integrado
print(f"Total de registros integrados: {len(df_final)}")
print("\nEstructura de las primeras filas indexadas:")
print(df_final[['id_siniestro', 'fecha', 'tipo_de_via_siniestro', 'modo_transporte', 'sexo_victima']].head())

##### 1.1 Diagnóstico de la Integración de Datos

La unión interna produjo 62 076 registros. Este volumen se explica por la granularidad de la tabla "VÍCTIMAS", donde varias víctimas pueden estar vinculadas a un mismo hecho. Los valores faltantes (SD) fueron convertidos a valores nulos (NaN) para garantizar la fiabilidad de los análisis posteriores.

### 3. ESTANDARIZACIÓN DE VARIABLES

In [ ]:
# Homogeneización nombres
df_final = df_final.rename(columns={
    'GRAVEdad_victima': 'gravedad_victima',
    'mes_siniestro': 'mes',
    'dia_siniestro': 'dia',
    'comuna_siniestro': 'comuna',
    'tipo_de_via_siniestro': 'tipo_via'
})

# Normalización de registros categóricos para prevenir duplicaciones
df_final['modo_transporte'] = df_final['modo_transporte'].str.upper().str.strip()
df_final['sexo_victima'] = df_final['sexo_victima'].str.upper().str.strip()

# Verificación del esquema definitivo
print("Esquema de datos normalizado y depurado:")
print(df_final.columns.tolist())

##### 1.2 Homogeneización Sintáctica y Normalización Categorizada

Se estandarizaron las etiquetas del dataset: corrección de mayúsculas/minúsculas, formato `snake_case` y conversión de las variables `modo_transporte` y `sexo_victima` a mayúsculas. Esto evita duplicados artificiales en los análisis posteriores.

### 4. ANÁLISIS DESCRIPTIVO: DISTRIBUCIÓN POR DÍAS DE LA SEMANA

In [ ]:
# Formato datetime
df_final['fecha'] = pd.to_datetime(df_final['fecha'], errors='coerce')

# Extraer el número del día de la semana
df_final['dia_semana_num'] = df_final['fecha'].dt.dayofweek

# Transformar los números en nombres de días
mapeo_dias = {
    0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves',
    4: 'Viernes', 5: 'Sábado', 6: 'Domingo'
}
df_final['dia_semana'] = df_final['dia_semana_num'].map(mapeo_dias)

# Contabilizar la frecuencia de siniestros por día de la semana
accidentes_por_dia_semana = df_final['dia_semana'].value_counts().reset_index()
accidentes_por_dia_semana.columns = ['dia_semana', 'cantidad_siniestros']

print("--- Días más peligrosos de la semana ---")
print(accidentes_por_dia_semana)

##### 1.3 Análisis de Siniestralidad según la Cronología Semanal

La distribución por día de la semana permite identificar patrones de movilidad: los días laborales concentran la siniestralidad relacionada con el tránsito comercial, mientras que el fin de semana refleja dinámicas de ocio nocturno. Esta variable es clave para planificar la presencia preventiva de control vial.

### 5. ANÁLISIS DESCRIPTIVO: PERFIL DEMOGRÁFICO DE LAS VÍCTIMAS

In [ ]:
# Conversión de la edad a tipo numérico
df_final['edad_victima'] = pd.to_numeric(df_final['edad_victima'], errors='coerce')

# Agregación por género y severidad
profil_victimes = df_final.groupby(['sexo_victima', 'gravedad_siniestro']).agg(
    edad_promedio=('edad_victima', 'mean'),
    total_casos=('id_siniestro', 'size')
).reset_index()

print("\n--- Perfil demográfico de las víctimas ---")
print(profil_victimes)

##### 1.4 Caracterización Demográfica y Matriz de Vulnerabilidad

El cruce de sexo, edad y gravedad revela dos perfiles de riesgo diferenciados:

- **Hombres:** concentran la mayoría de los casos graves y mortales (~73% de los decesos), con una edad promedio de 40 años. Su sobreexposición está ligada al transporte de cargas y la motomensajería.
- **Mujeres:** aunque representan menos casos, su edad promedio en accidentes mortels es de 51 años, lo que sugiere una mayor vulnerabilidad peatonal en edades avanzadas.

### 6. EXTRACCIÓN CRONOLÓGICA Y HORAS PICO

In [ ]:
def extraer_hora(val):
    """
    Función de conversión personalizada para homogeneizar el formato de hora.
    Soporta cadenas de texto (str), objetos datetime/time y maneja nulos.
    """
    if pd.isna(val):
        return np.nan
    # Procesamiento si el registro es una cadena de texto (ej. "13:40:00")
    if isinstance(val, str):
        try:
            return int(val.split(':')[0])
        except:
            return np.nan
    # Procesamiento si el registro ya es un objeto de tipo tiempo/fecha
    if hasattr(val, 'hour'):
        return val.hour
    return np.nan

# Aplicación de la función
df_final['hora'] = df_final['hora_siniestro'].apply(extraer_hora)

# Evaluación de Ventanas Críticas de Demanda (Horas Pico AM/PM)
pico_manana = [7, 8, 9]
pico_tarde = [16, 17, 18]

accidents_matin = df_final[df_final['hora'].isin(pico_manana)].shape[0]
accidents_soir = df_final[df_final['hora'].isin(pico_tarde)].shape[0]

print("--- Análisis de horas pico (Conversión validada) ---")
print(f"Siniestros en hora pico matutina (07:00 - 09:59): {accidents_matin}")
print(f"Siniestros en hora pico vespertina (16:00 - 18:59): {accidents_soir}")
print(f"\nEficiencia de la conversión: {df_final['hora'].notna().sum()} / {len(df_final)} registros procesados.")

##### 1.5 Dinámica de la Siniestralidad en Horas Pico

La franja vespertina (16:00–18:59) registra **un 62% más de accidentes** que la mañana (07:00–09:59). Este desequilibrio responde a la acumulación de fatiga, menor visibilidad y mayor superposición de flujos (transporte público, cargas y viajes personales) en el retorno al hogar.

# 2. Modelado Espacial Global y Zonificación Operativa

### 7. IMPORTACIÓN DE LIBRERÍAS DE GEOPROCESAMIENTO

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

### 8. CONSTRUCCIÓN DEL GEODATAFRAME Y CARTOGRAFÍA BASE

In [ ]:
# Remoción de registros
df_geo = df_final.dropna(subset=['longitud_siniestro', 'latitud_siniestro']).copy()

# Instanciación del GeoDataFrame WGS84 (EPSG:4326)
gdf_siniestros = gpd.GeoDataFrame(
    df_geo,
    geometry=gpd.GeoSeries.from_xy(df_geo['longitud_siniestro'], df_geo['latitud_siniestro']),
    crs="EPSG:4326"
)

# Control de consistencia geoespacial
print(f"Tipo de objeto estructurado: {type(gdf_siniestros)}")
print(f"Sistema de Referencia de Coordenadas (CRS): {gdf_siniestros.crs}")

# Renderizado Cartográfico Base
fig, ax = plt.subplots(figsize=(10, 10))
gdf_siniestros.plot(ax=ax, alpha=0.3, markersize=3, color='red', label='Siniestros registrados')

# Ajustes estéticos del mapa base
ax.set_title("Distribución Espacial Absoluta de Siniestros Viales en CABA", fontsize=12, fontweight='bold')
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.grid(True, linestyle='--', alpha=0.3)

plt.show()

##### 2.1 Vectorización Geográfica y Visualización de la Densidad Base

Los datos se transforman en un objeto geoespacial bajo el sistema WGS84 (`EPSG:4326`). La visualización general de todos los puntos de accidente traza orgánicamente la estructura vial de CABA, evidenciando la necesidad de aplicar algoritmos de clustering para identificar las zonas de mayor concentración de riesgo.

### 9. CARTOGRAFÍA INTERACTIVA: ANÁLISIS EXPLICATIVO DE SEVERIDAD

In [ ]:
# Muestra aleatoria
echantillon_carte = gdf_siniestros.sample(500, random_state=42)

# Remoción de registros
echantillon_propre = echantillon_carte.dropna(subset=['gravedad_siniestro']).copy()

def determinar_color_severidad(gravedad):
    """
    Asigna un código de color hexadecimal en función de la gravedad del siniestro.
    Prioriza la visibilidad de los eventos críticos (graves y mortales).
    """
    if gravedad == 'LEVE':
        return '#2ec4b6'   # Verde agua (Bajo impacto)
    if gravedad == 'GRAVE':
        return '#ff9f1c'   # Naranja (Impacto medio)
    if gravedad == 'MORTAL':
        return '#e71d36'  # Rojo (Impacto crítico)
    return '#7209b7'      # Violeta (Categoría residual)

# Mapeo de la paleta cromática sobre el subset de datos
echantillon_propre['color_diseno'] = echantillon_propre['gravedad_siniestro'].apply(determinar_color_severidad)

# Renderizado del mapa dinámico interactivo
mapa_interactivo = echantillon_propre.explore(
    color=echantillon_propre['color_diseno'],
    tiles="CartoDB positron",
    tooltip=['id_siniestro', 'modo_transporte', 'sexo_victima', 'gravedad_siniestro'],
    marker_kwds=dict(radius=5, fill=True)
)

mapa_interactivo

##### 2.2 Microanálisis Cartográfico e Interactividad Dinámica

Se visualiza una muestra de 500 registros clasificados por color: verde (leve), naranja (grave), rojo (mortal). Los puntos críticos se concentran en nodos de intermodalidad (Retiro, Constitución, accesos a autopistas), confirmando que la peligrosidad está asociada a la velocidad de diseño de las vías.

### 10. FILTRADO ESTRATÉGICO: DATASET DE SINIESTRALIDAD PRIORITARIA

In [ ]:
# Registros críticos
df_prioritaire = gdf_siniestros[gdf_siniestros['gravedad_siniestro'].isin(['GRAVE', 'MORTAL'])].copy()

def asignar_color_prioritario(gravedad):
    """
    Asigna una codificación cromática binaria.
    """
    if gravedad == 'GRAVE':
        return '#ff9f1c'   # Naranja (Lesiones graves)
    if gravedad == 'MORTAL':
        return '#e71d36'  # Rojo (Fatalidades)
    return '#7209b7'

# Aplicación
df_prioritaire['couleur_accidente'] = df_prioritaire['gravedad_siniestro'].apply(asignar_color_prioritario)

# Renderizado del mapa de calor de crisis interactivo
mapa_prioritario = df_prioritaire.explore(
    color=df_prioritaire['couleur_accidente'],
    tiles="CartoDB positron",
    tooltip=['id_siniestro', 'modo_transporte', 'sexo_victima', 'gravedad_siniestro'],
    marker_kwds=dict(radius=4, fill=True)
)

mapa_prioritario

##### 2.3 Delimitación de la Base de Siniestralidad Prioritaria

Se eliminan los siniestros leves para trabajar únicamente con los casos **GRAVE** y **MORTAL** (3 056 registros). La nueva distribución espacial muestra una concentración lineal clara sobre las grandes arterias, validando la hipótesis de que la severidad está directamente vinculada a la velocidad de circulación.

### 11. LIMPIEZA WKT Y PROYECCIÓN MÉTRICA

In [ ]:
import re
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point

# 1. Filtrado de registros con datos de georreferenciación plana
df_geo_avanced = df_final.dropna(subset=['geocodificacion_plana']).copy()

def corregir_wkt_puntos(texto_wkt):
    """
    Función de limpieza mediante expresiones regulares (Regex).

    """
    if not isinstance(texto_wkt, str):
        return None
    try:
        # Extracción de secuencias numéricas dentro de la estructura WKT
        numeros = re.findall(r"[-+]?\d*\.\d+|\d+", texto_wkt)

        nuevos_numeros = []
        for num in numeros:
            # Corrección de anomalías de doble punto (ej. 100.937.5 -> 100937.5)
            if num.count('.') > 1:
                partes = num.split('.')
                entero = "".join(partes[:-1])
                decimal = partes[-1]
                num = f"{entero}.{decimal}"
            nuevos_numeros.append(num)

        # Reconstrucción de la geometría en formato estándar String
        return f"POINT ({nuevos_numeros[0]} {nuevos_numeros[1]})"
    except:
        return None

# Aplicación
df_geo_avanced['geocodificacion_plana_limpia'] = df_geo_avanced['geocodificacion_plana'].apply(corregir_wkt_puntos)
df_geo_avanced = df_geo_avanced.dropna(subset=['geocodificacion_plana_limpia'])

# 3. Parsed de cadenas WKT a objetos geométricos nativos de Shapely
df_geo_avanced['geometry'] = df_geo_avanced['geocodificacion_plana_limpia'].apply(wkt.loads)

# 4. Definición paramétrica del Sistema de Proyección Cartográfica (PROJ4) adaptado para CABA
crs_caba_proyectado = "+proj=tmerc +lat_0=-34.6297166 +lon_0=-58.4627 +k=1 +x_0=100000 +y_0=100000 +ellps=WGS84 +units=m +no_defs"

# Instanciación del GeoDataFrame Métrico Proyectado
gdf_caba_metrico = gpd.GeoDataFrame(df_geo_avanced, geometry='geometry', crs=crs_caba_proyectado)

print("--- Represenación Métrique Reparada (Gauss-Kruger CABA) ---")
print(gdf_caba_metrico[['id_siniestro', 'geometry']].head(2))

# Cálculo de Distancia Euclídea Coaxial (Centro de Gravedad: Obelisco)
obelisco_point = Point(107530.0, 103220.0)

# Cálculo indexado de distancias reales expresadas en kilómetros
gdf_caba_metrico['distancia_obelisco_km'] = gdf_caba_metrico['geometry'].distance(obelisco_point) / 1000

print("\n--- Variable espacial creada con éxito ---")
print(gdf_caba_metrico[['id_siniestro', 'distancia_obelisco_km']].head())

# Reproyección ortogonal al sistema geográfico global EPSG:4326 para interoperabilidad
gdf_siniestros = gdf_caba_metrico.to_crs(epsg=4326)
print("\n Reproyección exitosa al estándar global WGS84 (EPSG:4326).")

##### 2.4 Fundamentos de Geodesia: Coordenadas Planas (Gauss-Kruger) vs. Geográficas (WGS84)


Para calcular distancias reales en metros, los datos se reproyectan al sistema métrico local. Esto permite obtener la variable `distancia_obelisco_km` con precisión geométrica, evitando las distorsiones propias del cálculo en grados decimales.

### 12. ANÁLISIS EXPLORATORIO MULTIVARIADO (FACET GRID POR MODO DE TRANSPORTE)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

import seaborn as sns
import matplotlib.pyplot as plt

# 1. Filtrado categorías principales de transporte
principaux_transports = ['MOTO', 'AUTO', 'PEATON', 'BICICLETA']
df_filtre = gdf_siniestros[gdf_siniestros['modo_transporte'].isin(principaux_transports)]

# 2. Configuración de las colores
colores = {'MOTO': 'purple', 'AUTO': 'blue', 'PEATON': 'orange', 'BICICLETA': 'green'}

# 3. Función carta
def carte_specifique(data, **kwargs):
    ax = plt.gca()
    modo = data['modo_transporte'].iloc[0]

    # Graficado de los puntos georreferenciados
    data.plot(ax=ax, alpha=0.2, markersize=2, color=colores.get(modo, 'red'))

    # Restricción estricta de límites coordenados
    ax.set_xlim(-58.55, -58.33)
    ax.set_ylim(-34.70, -34.52)

    # Remoción de ejes
    ax.set_axis_off()

# 4. Instanciación de la grilla múltiple FacetGrid
g = sns.FacetGrid(df_filtre, col="modo_transporte", col_wrap=2, height=5)

# 5. Mapeo de la función cartográfica
g.map_dataframe(carte_specifique)

# 6. Ajustes de títulos y formato
g.set_titles(col_template="{col_name}")
plt.subplots_adjust(top=0.9)
g.fig.suptitle('Distribución Espacial de Siniestros por Modo de Desplazamiento en CABA', fontsize=14, fontweight='bold')

plt.show()

### 2.5 Análisis Espacial Comparativo por Tipología de Actor Vial

El mapa FacetGrid desglosa la siniestralidad por tipo de usuario:

- **MOTO:** distribución uniforme en toda la ciudad — el riesgo es ubiquitario.
- **AUTO:** concentrado en autopistas periféricas (Gral. Paz, Lugones).
- **PEATÓN:** polarizado en el centro y grandes nodos de transbordo (Retiro, Constitución).
- **BICICLETA:** presente a lo largo de corredores residenciales con escasa infraestructura dedicada.

### 13. SEGMENTACIÓN ESPACIAL ÓPTIMA (RUPTURAS NATURALES DE JENKS)

In [ ]:
import mapclassify as mc
import pandas as pd

# 1. Aplicación del clasificador de Jenks para identificar discontinuidades óptimas en la distancia
classifier = mc.NaturalBreaks(gdf_siniestros['distancia_obelisco_km'], k=3)

# 2. Asignación de etiquetas morfológicas
# El orden corresponde a los límites determinados estadísticamente por el modelo
labels_zonas = ['Centro / Radio Urbano Denso', 'Macrocentro / Eje Intermedio', 'Periferia / Corredores de Alta Velocidad']
gdf_siniestros['zona_distancia'] = gdf_siniestros['distancia_obelisco_km'].apply(lambda x: labels_zonas[classifier(x)[0]])

# 3. Construcción de la matriz de contingencia normalizada por filas (%)
print("--- Análisis del impacto espacial en la gravedad (Método de Jenks) ---")
matrice_gravite_zone = pd.crosstab(
    gdf_siniestros['zona_distancia'],
    gdf_siniestros['gravedad_siniestro'],
    normalize='index'
) * 100

print(matrice_gravite_zone.round(2))

##### 2.6 Estratificación Espacial mediante Rupturas Naturales de Jenks

El algoritmo de Jenks divide el territorio en tres coronas concéntricas de riesgo (centro, macrocentro, periferia). Los resultados revelan que la zona intermedia (3–8 km del Obelisco) concentra el mayor índice de mortalidad (1,32%), mientras que la periferia registra una tasa de mortales estadísticamente nula por el alto volumen de colisiones leves en horas pico.

### 14. AGREGACIÓN ESPACIAL POR UNIDADES ADMINISTRATIVAS (COMUNAS)

In [ ]:
import pandas as pd

# 1. Filtrado estricto para aislar siniestros de máxima severidad (Graves y Mortales)
df_graves = gdf_siniestros[gdf_siniestros['gravedad_siniestro'].isin(['GRAVE', 'MORTAL'])]

# 2. Agregación y conteo de frecuencias por unidad poligonal de Comuna
comunas_danger = df_graves.groupby('comuna').size().reset_index(name='nombre_accidents_graves')

# 3. Ordenamiento descendente
comunas_danger = comunas_danger.sort_values(by='nombre_accidents_graves', ascending=False)

print("--- Ranking de Comunas por tasa de accidentalidad grave ---")
print(comunas_danger.head(5))

##### 2.7 Análisis Jurisdiccional: Distribución Poligonal por Comunas

Al agrupar los siniestros prioritarios por Comuna, la **Comuna 1** lidera con 332 accidentes graves, seguida por las Comunas 12, 14 y 13. Estas jurisdicciones concentran los mayores flujos intermodales de la ciudad (terminales, subte, buses).

### 15. MODELADO DE DENSIDAD ESPACIAL GLOBAL (DBSCAN)

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN

# 1. Transformación angular
coordenadas_radianes = np.radians(df_prioritaire[['latitud_siniestro', 'longitud_siniestro']].values)

# --- Parametrización del Algoritmo (Justificación de Umbrales) ---
# Radio máximo de vecindad expresado en kilómetros (100 metros)
radio_km = 0.1
# Conversión del radio métrico al equivalente angular en radianes (Radio terrestre medio: 6371.0088 km)
epsilon_radian = radio_km / 6371.0088

# Umbral de masa crítica: número mínimo de siniestros prioritarios
min_siniestros_umbral = 8

# 2. Instanciación y entrenamiento del modelo DBSCAN con métrica geodésica
db_global = DBSCAN(eps=epsilon_radian, min_samples=min_siniestros_umbral, metric='haversine')
df_prioritaire['cluster_dbscan'] = db_global.fit_predict(coordenadas_radianes)

# 3. Extracción de métricas de diagnóstico del modelo
labels_globales = df_prioritaire['cluster_dbscan']
num_clusters = len(set(labels_globales)) - (1 if -1 in labels_globales else 0)
puntos_en_clusters = df_prioritaire[df_prioritaire['cluster_dbscan'] != -1].shape[0]

print("--- DIAGNÓSTICO DEL MODELADO DE DENSIDAD DBSCAN ---")
print(f"Radio operativo establecido (Epsilon): {radio_km * 1000} metros")
print(f"Masa crítica mínima por nodo (Min Samples): {min_siniestros_umbral} siniestros")
print(f"Cantidad de puntos críticos densos detectados: {num_clusters}")
print(f"Volumen de siniestros graves/mortales absorbidos por los clusters: {puntos_en_clusters} de {len(df_prioritaire)}")

### 2.8 Configuración y Diagnóstico del Modelo de Densidad Global DBSCAN

DBSCAN detecta zonas de alta densidad sin imponer un número fijo de clusters, y etiqueta como "ruido" los accidentes aislados. Se calibró con `ε = 100 m` (escala de una cuadra) y `min_samples = 8`, lo que permite identificar intersecciones con acumulación estructural de accidentes graves, descartando eventos aislados o fortuitos.

### 16. ESCENARIO A: CALIBRACIÓN ULTRA-CRÍTICA (DBSCAN QUIRÚRGICO)

In [ ]:
# --- Parametrización Restrictiva de Alta Densidad ---
# Radio máximo de vecindad expresado en kilómetros (100 metros)
radio_km = 0.05
# Reducción del radio operativo a 50 metros
epsilon_radian = radio_km / 6371.0088

# Incremento a 10 siniestros graves
min_siniestros = 10

# Entrenando el modelo bajo el escenario restrictivo
db = DBSCAN(eps=epsilon_radian, min_samples=min_siniestros, metric='haversine')
df_prioritaire['cluster_dbscan'] = db.fit_predict(coordenadas_radianes)

# Extracción de métricas de diagnóstico del escenario
labels_globales = df_prioritaire['cluster_dbscan']
num_clusters_q = len(set(labels_globales)) - (1 if -1 in labels_globales else 0)
puntos_en_clusters_q = df_prioritaire[df_prioritaire['cluster_dbscan'] != -1].shape[0]

print("--- RÉSULTATS DU CLUSTERING DBSCAN (ESCENARIO A) ---")
print(f"Radio operativo estricto (Epsilon): {radio_km * 1000} metros")
print(f"Masa crítica mínima por nodo (Min Samples): {min_siniestros} siniestros")
print(f"Cantidad de puntos negros ultra-críticos detectados: {num_clusters_q}")
print(f"Volumen de siniestros graves absorbidos bajo este escenario: {puntos_en_clusters_q} de {len(df_prioritaire)}")

##### 2.9 Análisis de Sensibilidad: Escenario A (Enfoque Quirúrgico de Alta Densidad)

Con un radio ultra-reducido, el modelo identifica solo **8 intersecciones** de máxima concentración. Son los puntos más peligrosos de toda la ciudad, ideales para intervenciones de ingeniería vial inmediata (rediseño semafórico, canalización de giros). Limitación: captura solo el 3% de los accidentes graves.

### 17. ESCENARIO B: CALIBRACIÓN MACRO (DBSCAN DE PLANIFICACIÓN BARRIAL)

In [ ]:
# --- Parametrización Extensiva para Planificación Urbana ---
# Incremento del radio de vecindad a 250 metros para capturar el entorno del barrio
radio_barrial_km = 0.25
epsilon_radian_barrial = radio_barrial_km / 6371.0088

# Ajuste de la masa crítica mínima a 15 siniestros prioritarios
min_siniestros_barrial = 15

# Entrenando el modelo bajo el escenario macro
db_barrial = DBSCAN(eps=epsilon_radian_barrial, min_samples=min_siniestros_barrial, metric='haversine')
df_prioritaire['cluster_dbscan_barrial'] = db_barrial.fit_predict(coordenadas_radianes)

# Extracción de métricas de diagnóstico del escenario
labels_barriales = df_prioritaire['cluster_dbscan_barrial']
num_clusters_b = len(set(labels_barriales)) - (1 if -1 in labels_barriales else 0)
puntos_en_clusters_b = df_prioritaire[df_prioritaire['cluster_dbscan_barrial'] != -1].shape[0]

print("--- RÉSULTATS DU CLUSTERING DBSCAN (ESCENARIO B) ---")
print(f"Radio operativo ampliado (Epsilon): {radio_barrial_km * 1000} metros")
print(f"Masa crítica mínima por zona (Min Samples): {min_siniestros_barrial} siniestros")
print(f"Cantidad de zonas críticas barriales detectadas: {num_clusters_b}")
print(f"Volumen de siniestros graves absorbidos bajo este escenario: {puntos_en_clusters_b} de {len(df_prioritaire)}")

##### 2.10 Análisis de Sensibilidad: Escenario B (Enfoque Macrorregional de Planificación Barrial)

Con un radio ampliado, el modelo captura **508 accidentes** (16%) agrupando tramos enteros de avenidas o zonas con problemas sistémicos. Es útil para planificar intervenciones de mediano plazo (pacificación del tránsito, mejora de ciclovías), pero pierde precisión a nivel de intersección.

### 18. VISUALIZACIÓN INTERACTIVA DEL ESCENARIO DE MÁXIMA SEVERIDAD (PUNTOS NEGROS)

In [ ]:
# 1. Ajuste definitivo del modelo ideal
db_ideal = DBSCAN(eps=0.1/6371.0088, min_samples=8, metric='haversine')
df_prioritaire['cluster_label'] = db_ideal.fit_predict(coordenadas_radianes)

# 2. Generación de atributos cromáticos y semánticos para el renderizado
# El ruido (-1) se atenúa en gris; los puntos negros densos se destacan en rojo crítico
df_prioritaire['couleur_cluster'] = df_prioritaire['cluster_label'].apply(
    lambda x: '#9ca3af' if x == -1 else '#e63946'
)

# Definición de etiquetas descriptivas para el tooltip interactivo
df_prioritaire['statut_point'] = df_prioritaire['cluster_label'].apply(
    lambda x: 'Siniestro Aislado (Ruido)' if x == -1 else f'PUNTO NEGRO CRÍTICO (Cluster N°{x})'
)

# 3. Renderizado cartográfico dinámico con diferenciación geométrica por densidad
mapa_puntos_negros = df_prioritaire.explore(
    color=df_prioritaire['couleur_cluster'],
    tiles="CartoDB positron",
    style_kwds={
        "style_function": lambda x: {
            "radius": 2.5 if x["properties"]["cluster_label"] == -1 else 6,
            "fill": True
        }
    },
    tooltip=['statut_point', 'modo_transporte', 'gravedad_siniestro', 'comuna']
)

mapa_puntos_negros

##### 2.11 Cartografía de Alta Resolución y Aislamiento de Puntos Negros Críticos

La visualización final del escenario estándar (ε = 100 m) aísla **24 puntos negros** sobre el mapa. Los accidentes en gris representan riesgo disperso (no estructural), mientras que los puntos rojos marcan las intersecciones con falla sistemática de infraestructura. Esta cartografía es directamente operativa para auditorías de seguridad vial.

### 19. MODELADO PARTISIONAL: ZONIFICACIÓN OPERATIVA MEDIANTE K-MEANS

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd

# 1. Extracción de la matriz de características en coordenadas espaciales absolutas
X_kmeans = df_prioritaire[['longitud_siniestro', 'latitud_siniestro']].values

# 2. Instanciación del modelo K-Means parametrizado para 24 regiones operativas
# Se fija n_init para garantizar la convergencia óptima de los centroides iniciales
kmeans_sectores = KMeans(n_clusters=24, random_state=42, n_init=10)
df_prioritaire['cluster_kmeans'] = kmeans_sectores.fit_predict(X_kmeans)

# 3. Conversión de las etiquetas numéricas a tipo cadena para optimizar el mapeo categórico
df_prioritaire['cluster_kmeans_str'] = df_prioritaire['cluster_kmeans'].astype(str)

print("--- DIAGNÓSTICO DE LA ZONIFICACIÓN K-MEANS ---")
print(f"Algoritmo K-Means convergido con éxito para K = 24 regiones.")
print(f"Siniestros calificados como residuo (ruido): 0 de {len(df_prioritaire)}")
print("Nota operativa: K-Means impone un particionamiento exhaustivo sobre la totalidad del espacio sampleado.")

# 4. --- Renderizado Cartográfico del Particionamiento Regional ---
mapa_kmeans = df_prioritaire.explore(
    column='cluster_kmeans_str',
    cmap='tab20',
    tiles="CartoDB positron",
    style_kwds={"style_function": lambda x: {"radius": 4, "fill": True}},
    tooltip=['cluster_kmeans', 'modo_transporte', 'comuna']
)

mapa_kmeans

##### 2.12 Regionalización Exhaustiva y Límites del Particionamiento K-Means

K-Means divide la ciudad en 24 sectores de patrullaje cubriendo el 100% de los accidentes. A diferencia de DBSCAN, no existe "ruido": todos los eventos quedan asignados a un sector. Su principal límite es que genera regiones geométricas rígidas, sin considerar la morfología vial real ni la densidad diferenciada del riesgo.

# 3. Comparación de Modelos e Impacto Operativo

### 20. ANÁLISIS DE SENSIBILIDAD ESPACIO-TEMPORAL (NOCHE VS. HORAS PICO)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

# 1. Construcción de máscaras temporales indexadas
mask_noche = (df_prioritaire['hora'] >= 22) | (df_prioritaire['hora'] < 6)
mask_pico = ((df_prioritaire['hora'] >= 7) & (df_prioritaire['hora'] <= 10)) | \
            ((df_prioritaire['hora'] >= 17) & (df_prioritaire['hora'] <= 20))

df_noche = df_prioritaire[mask_noche].copy()
df_pico = df_prioritaire[mask_pico].copy()

# 2. Extracción de componentes angulares en radianes para cálculo Haversine
coordenadas_noche = np.radians(df_noche[['latitud_siniestro', 'longitud_siniestro']].values)
coordenadas_pico = np.radians(df_pico[['latitud_siniestro', 'longitud_siniestro']].values)

# 3. Parametrización y entrenamiento de los modelos DBSCAN específicos por ventana horaria
# Se utilizan los umbrales óptimos validados (50 metros y masa crítica de 10)
db_noche = DBSCAN(eps=0.05/6371.0088, min_samples=10, metric='haversine')
db_pico = DBSCAN(eps=0.05/6371.0088, min_samples=10, metric='haversine')

df_noche['cluster_temporal'] = db_noche.fit_predict(coordenadas_noche)
df_pico['cluster_temporal'] = db_pico.fit_predict(coordenadas_pico)

# 4. Extracción de métricas de diagnóstico comparativo
labels_noche = df_noche['cluster_temporal']
labels_pico = df_pico['cluster_temporal']

num_clusters_noche = len(set(labels_noche)) - (1 if -1 in labels_noche else 0)
num_clusters_pico = len(set(labels_pico)) - (1 if -1 in labels_pico else 0)

print("=========================================================================")
print("   ANÁLISIS TEMPORAL AVANZADO: SENSIBILIDAD DEL DBSCAN (NOCHE VS HORA PICO) ")
print("=========================================================================")
print(f"Siniestros analizados en horario Nocturno (22:00 a 05:59): {df_noche.shape[0]}")
print(f"-> Puntos negros críticos detectados en la noche: {num_clusters_noche}")
print(f"-> Siniestros concentrados en puntos negros nocturnos: {df_noche[df_noche['cluster_temporal'] != -1].shape[0]}")
print("-------------------------------------------------------------------------")
print(f"Siniestros analizados en Horas Pico (07:00-10:59 / 17:00-20:59): {df_pico.shape[0]}")
print(f"-> Puntos negros críticos detectados en horas pico: {num_clusters_pico}")
print(f"-> Siniestros concentrados en puntos negros de horas pico: {df_pico[df_pico['cluster_temporal'] != -1].shape[0]}")
print("=========================================================================")

##### 3.1 Análisis de Sensibilidad Dinámica: El Factor Espacio-Temporal

Al aplicar DBSCAN sobre dos ventanas temporales separadas:
- **Horas pico:** el riesgo se agrupa en nodos fijos predecibles (saturación de intersecciones).
- **Horario nocturno:** la mayoría de los accidentes se clasifica como "ruido" — el riesgo nocturno es disperso y responde a factores dinámicos (velocidad, fatiga, visibilidad reducida).

**Conclusión operativa:** el patrullaje diurno debe concentrarse en nodos fijos; el nocturno debe ser móvil y orientado a corredores de alta velocidad.

In [ ]:
print(df_prioritaire.columns.tolist())

### 21. PERFILADO DE INFRAESTRUCTURA POR SECTOR LOGÍSTICO (K-MEANS)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Construcción de la matriz de composición de infraestructura vial por sector operativo
# Se normaliza por fila para evaluar la distribución relativa (%) de cada tipología de vía
tabla_perfiles = pd.crosstab(
    df_prioritaire['cluster_kmeans'],
    df_prioritaire['tipo_via'],
    normalize='index'
) * 100

# 2. --- Renderizado del Gráfico de Barras Apiladas (Stacked Bar Chart) ---
fig, ax = plt.subplots(figsize=(16, 8))
tabla_perfiles.plot(kind='bar', stacked=True, cmap='viridis', ax=ax)

# 3. Optimización paramétrica y diseño estético del reporte gráfico
ax.set_title("Perfilado de Riesgo por Tipología de Vía en Sectores K-Means", fontsize=14, fontweight='bold')
ax.set_xlabel("Sectores Operativos K-Means (ID Región)", fontsize=12)
ax.set_ylabel("Distribución Porcentual del Siniestro (%)", fontsize=12)
plt.xticks(rotation=0)

# Posicionamiento estratégico de la leyenda fuera del área de trazado
ax.legend(title="Configuración de la Infraestructura Vía", bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

##### 3.2a Perfilado de Infraestructura y Tipificación de Sectores Logísticos

El análisis de los 24 sectores K-Means según el tipo de vía permite definir estrategias de intervención específicas:
- **Sectores de avenidas (>70% avenidas):** control de radares y sincronización semafórica.
- **Sectores de calles secundarias:** pacificación del tránsito (lomos de burro, sendas elevadas).
- **Sectores periféricos con autopistas:** control de velocidad y fotomultas.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Construcción de la matriz de composición de actores viales por sector operativo
# Se normaliza por fila para evaluar la distribución relativa (%) del modo de transporte
tabla_perfiles = pd.crosstab(
    df_prioritaire['cluster_kmeans'],
    df_prioritaire['modo_transporte'],
    normalize='index'
) * 100

# 2. --- Renderizado del Gráfico de Barras Apiladas (Stacked Bar Chart) ---
fig, ax = plt.subplots(figsize=(16, 8))
tabla_perfiles.plot(kind='bar', stacked=True, cmap='viridis', ax=ax)

# 3. Optimización paramétrica y diseño estético del reporte gráfico
ax.set_title("Perfilado de Riesgo por Modo de Transporte en Sectores K-Means", fontsize=14, fontweight='bold')
ax.set_xlabel("Sectores Operativos K-Means (ID Región)", fontsize=12)
ax.set_ylabel("Distribución Porcentual del Siniestro (%)", fontsize=12)
plt.xticks(rotation=0)

# Posicionamiento estratégico de la leyenda fuera del área de trazado
ax.legend(title="Modo de Transporte / Actor Vial", bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 3.2b Perfil de Actores Viales por Sector

Cada sector K-Means tiene una "firma de riesgo" diferente según el tipo de usuario:
- **Predominio de PEATÓN** (ej. Sector 20 – Comuna 1): intervenciones de infraestructura peatonal, fases semafóricas exclusivas.
- **Predominio de MOTO:** fiscalización de cascos, velocidad y carriles de carga.
- **Perfil mixto:** patrullaje móvil de control de velocidad general.

### 22. CORRELACIÓN JURISDICCIONAL: ASIGNACIÓN DE SECTORES POR COMUNA PRINCIPAL

In [ ]:
# import pandas as pd

# 1. Construcción de la matriz de contingencia cruzada entre sectores y comunas
# Se normaliza por fila para identificar la concentración relativa por jurisdicción
tabla_comunas = pd.crosstab(
    df_prioritaire['cluster_kmeans'],
    df_prioritaire['comuna'],
    normalize='index'
) * 100

# 2. --- Extracción Indexada de Predominancia Jurisdiccional ---
print("=========================================================")
print("   LOCALIZACIÓN DE SECTORES POR COMUNA PREDOMINANTE")
print("=========================================================")
for cluster in sorted(df_prioritaire['cluster_kmeans'].unique()):
    comuna_predominante = tabla_comunas.loc[cluster].idxmax()
    porcentaje = tabla_comunas.loc[cluster].max()
    print(f"Sector Operativo {cluster:02d} -> Asignación principal: COMUNA {comuna_predominante:<2} ({porcentaje:>5.1f}% de los siniestros)")
print("=========================================================")

##### 3.3 Asignación de Sectores por Comuna

Al cruzar los sectores K-Means con las Comunas, se obtiene la jurisdicción responsable de cada zona. Los sectores con alta concentración en una sola Comuna (>80%) permiten asignar responsabilidades claras. Los sectores límite (~50%) capturan corredores entre comunas (como la Av. General Paz) y requieren coordinación inter-jurisdiccional.

### 23. CARTOGRAFÍA ESTÁTICA REGIONAL: MAPA DE SECTORES K-MEANS

In [ ]:
import matplotlib.pyplot as plt

# Instanciación de la figura cartográfica estática para el reporte final
fig, ax = plt.subplots(figsize=(12, 12))

# Renderizado de los puntos de siniestralidad prioritarios clasificados por sector
df_prioritaire.plot(
    column='cluster_kmeans',
    categorical=True,
    legend=True,
    cmap='tab20',
    markersize=15,
    alpha=0.8,
    ax=ax,
    legend_kwds={
        'title': "Sectores Operativos K-Means",
        'bbox_to_anchor': (1.02, 1),
        'loc': 'upper left'
    }
)

# Ajustes de formato, grilla geodésica y etiquetas institucionales
ax.set_title("Distribución Geográfica de los 24 Sectores Operativos K-Means en CABA", fontsize=14, fontweight='bold')
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

##### 3.4 Cartografía K-Means

El mapa de los 24 sectores confirma la cobertura total del territorio. Las celdas se compactan en el centro (alta densidad de eventos) y se expanden en la periferia. Este mapa es complementario al de DBSCAN: mientras DBSCAN indica *dónde rediseñar la infraestructura*, K-Means define *cómo distribuir las patrullas*.

### 24. INDEXACIÓN ESPACIAL DISCRETA MEDIANTE SISTEMA DE GRILLAS H3 (UBER)

In [ ]:
!pip install h3

In [ ]:
import h3
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

# 1. Definición paramétrica de la resolución espacial (Res 8 ~ 0.7 km² de área por celda)
RESOLUTION = 8

# 2. Indexación adaptativa utilizando la API v4 de H3
df_prioritaire['h3_index'] = df_prioritaire.apply(
    lambda row: h3.latlng_to_cell(row['latitud_siniestro'], row['longitud_siniestro'], RESOLUTION),
    axis=1
)

# 3. Agregación y conteo de frecuencias absolutas por celda hexagonal
df_h3_counts = df_prioritaire.groupby('h3_index').size().reset_index(name='cantidad_siniestros')

# Identificación del top de celdas con mayor densidad de riesgo prioratario
top_10_hexagonos = df_h3_counts.sort_values(by='cantidad_siniestros', ascending=False).head(10)

print("=========================================================")
print(f"   TOP 10 DE HEXÁGONOS MÁS PELIGROSOS EN CABA (RES {RESOLUTION})")
print("=========================================================")
for idx, row in enumerate(top_10_hexagonos.itertuples(), 1):
    print(f"Top {idx:02d}: Celda Hexagonal {row.h3_index} -> {row.cantidad_siniestros:>4} siniestros graves/mortales")
print("=========================================================")

# 4. --- Vectorización de Fronteras Hexagonales a Objetos Shapely ---
def h3_to_polygon(h3_id):
    """
    Transforma los vértices geométricos de una celda H3 en un objeto Polygon.
    Invierte el orden (lat, lon) de la salida nativa al estándar geográfico (lon, lat).
    """
    boundary = h3.cell_to_boundary(h3_id)
    return Polygon([(lon, lat) for lat, lon in boundary])

# Construcción e instanciación del GeoDataFrame indexado hexagonalmente
df_h3_counts['geometry'] = df_h3_counts['h3_index'].apply(h3_to_polygon)
gdf_h3 = gpd.GeoDataFrame(df_h3_counts, geometry='geometry', crs="EPSG:4326")

##### 3.5 Indexación Hexagonal H3 (Uber) – Resolución 8

La grilla H3 divide la ciudad en hexágonos regulares de ~0,7 km². Sus ventajas frente a las Comunas: fronteras objetivas (no políticas) y distancias equidistantes a los 6 vecinos. Cada hexágono permite calcular densidades de siniestros comparables e independientes de la morfología del barrio.

### 25. VISUALIZACIÓN CARTOGRÁFICA BICAPA: TOP 10 HEXÁGONOS CRÍTICOS H3

In [ ]:
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt

# Filtrado del GeoDataFrame para aislar espacialmente las 10 celdas del Top jerárquico
gdf_top10 = gdf_h3[gdf_h3['h3_index'].isin(top_10_hexagonos['h3_index'])]

fig, ax = plt.subplots(figsize=(12, 10))

# 1. Capa Base: Renderizado de la totalidad de las celdas H3 activas (atenuación espacial)
gdf_h3.plot(ax=ax, color='lightgrey', edgecolor='white', alpha=0.6)

# 2. Capa Temática: Superposición del Top 10 crítico indexado por escala de color coroplética
gdf_top10.plot(
    ax=ax,
    column='cantidad_siniestros',
    cmap='OrRd',
    edgecolor='black',
    linewidth=1.5,
    legend=True,
    legend_kwds={'label': "Frecuencia de Siniestros Prioritarios (Graves/Mortales)"}
)

# Optimización paramétrica
ax.set_title("Identificación Geométrica del Top 10 de 'Hexágonos Negros' H3 (Res 8)", fontsize=14, fontweight='bold')
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.grid(axis='both', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

##### 3.6 Top 10 Hexágonos Críticos

Los 10 hexágonos con mayor concentración de siniestros graves (entre 25 y 34 casos por celda) se localizan sobre la Av. General Paz y la zona de la Comuna 1/Retiro. Su visualización en escala de calor permite priorizar intervenciones en un perímetro perfectamente acotado de 0,7 km².

In [ ]:
!pip install osmnx

### 26. EXTRACCIÓN TOPOLÓGICA DE LA RED VIAL URBANA (OSMnX)

In [ ]:
import osmnx as ox
import geopandas as gpd

print("1. Descargando topología de la red vial de la Ciudad Autónoma de Buenos Aires...")
# Descarga del grafo vial optimizado para vehículos motorizados (drive)
G_caba = ox.graph_from_place("Buenos Aires, Argentina", network_type="drive")

print("2. Extracción y vectorización del GeoDataFrame de intersecciones (Nodos)...")
# Aislamiento exclusivo de la componente de nodos para evaluar la complejidad estructural
gdf_nodos_viales = ox.graph_to_gdfs(G_caba, nodes=True, edges=False)

# Garantizar la consistencia en el Sistema de Referencia de Coordenadas estándar WGS84
gdf_nodos_viales = gdf_nodos_viales.to_crs("EPSG:4326")

print(f"Descarga completada: {len(gdf_nodos_viales)} intersecciones viales indexadas en el espacio de análisis.")

##### 3.7 Red Vial Urbana con OSMnX

Se descarga la topología real de las calles de CABA mediante OSMnX, modelando la ciudad como un grafo dirigido. Los **17 910 nodos** extraídos representan intersecciones físicas reales. Este dataset permite cruzar los puntos negros con atributos de las esquinas (número de vías convergentes) para identificar si la complejidad geométrica explica la concentración de accidentes.

### 27. ANÁLISIS TOPOLÓGICO CRUZADO: SINIESTRALIDAD VS. COMPLEJIDAD DE RED

In [ ]:
import geopandas as gpd

# 1. Ejecución de la jointure espacial (Spatial Join) indexada
# Identifica en qué celda hexagonal H3 cae geométricamente cada intersección real
jointure_topologica = gpd.sjoin(gdf_nodos_viales, gdf_h3, how="inner", predicate="within")

# 2. Agregación estadística de los atributos del grafo urbano por celda H3
stats_graphe_urbano = jointure_topologica.groupby('h3_index').agg(
    total_intersections=('street_count', 'count'),
    complexite_moyenne=('street_count', 'mean')
).reset_index()

# 3. Fusión matricial con el Top 10 de máxima siniestralidad prioritaria
top_10_diagnostico_final = top_10_hexagonos.merge(stats_graphe_urbano, on='h3_index', how='left')

print("=========================================================================")
print("   DIAGNÓSTICO CRUZADO: SINIESTRALIDAD VS. ESTRUCTURA TOPOLÓGICA URBANA")
print("=========================================================================")
print(top_10_diagnostico_final[['h3_index', 'cantidad_siniestros', 'total_intersections', 'complexite_moyenne']]
      .rename(columns={
          'h3_index': 'ID Hexágono H3',
          'cantidad_siniestros': 'Siniestros Graves',
          'total_intersections': 'Total Esquinas',
          'complexite_moyenne': 'Complejidad Media (Vías)'
      }))
print("=========================================================================")

##### 3.8 Siniestralidad vs. Complejidad Topológica

Al cruzar los 10 hexágonos más peligrosos con la topología de OSMnX:
- **Densidad de esquinas:** entre 32 y 81 intersecciones por hexágono — zonas de alta fricción potencial.
- **Complejidad media:** entre 3,4 y 3,9 vías por nodo (próximo a 4, el valor de un cruce en cruz completo).

**Conclusión:** la siniestralidad grave no es aleatoria — tiene una causa física medible. Las zonas más peligrosas son aquellas con mayor densidad de cruces complejos, lo que señala la necesidad de rediseño geométrico (semaforización inteligente, canalización de giros, pacificación peatonal).

In [ ]:
print(gdf_siniestros['fecha'].dt.year.unique())